In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

orders_bronze = spark.table(
    "retail_bronze.orders"
)

print("Bronze order count:", orders_bronze.count())

display(orders_bronze.limit(10))

In [0]:
invalid_status_orders = (
    orders_bronze
    .filter(
        ~F.col("order_status").isin(
            "PLACED",
            "SHIPPED",
            "DELIVERED",
            "CANCELLED"
        )
    )
)

print(
    "Invalid status orders:",
    invalid_status_orders.count()
)

display(invalid_status_orders)

In [0]:
# Separate valid and invalid orders based on the allowed order statuses

valid_orders = (
    orders_bronze
    .filter(
        F.col("order_status").isin(
            "PLACED",
            "SHIPPED",
            "DELIVERED",
            "CANCELLED"
        )
    )
)

# Capture records with invalid order statuses for quarantine
invalid_orders = (
    orders_bronze
    .filter(
        ~F.col("order_status").isin(
            "PLACED",
            "SHIPPED",
            "DELIVERED",
            "CANCELLED"
        )
    )
)

# Validate the record counts after the quality check
print("Valid orders:", valid_orders.count())
print("Invalid orders:", invalid_orders.count())

In [0]:
# Load the validated customer IDs from the Silver layer
valid_customer_ids = (
    spark.table("retail_silver.customers")
    .select("customer_id")
    .distinct()
)

# Identify orders whose customer_id does not exist in the Customer dimension
orders_with_invalid_customer = (
    valid_orders
    .join(
        valid_customer_ids,
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Orders with invalid customer references:",
    orders_with_invalid_customer.count()
)

display(orders_with_invalid_customer.limit(20))

In [0]:
# Create test records with customer IDs that do not exist
# in the Silver Customers table

invalid_customer_orders = (
    spark.range(1, 11)
    .select(
        # Generate unique order IDs after the existing order range
        (F.col("id") + 100000).cast("long").alias("order_id"),

        # Deliberately use a non-existent customer ID
        F.lit(999999).cast("long").alias("customer_id"),

        # Use a valid order date
        F.current_date().alias("order_date"),

        # Use a valid order status so the record passes status validation
        F.lit("PLACED").alias("order_status"),

        # Generate a valid-looking order amount
        F.round(
            F.rand(seed=100) * 9950 + 50,
            2
        ).alias("total_amount"),

        # Record update timestamp
        F.current_timestamp().alias("updated_at")
    )
)

# Add the intentionally invalid records to our valid-status orders
valid_orders_with_bad_references = (
    valid_orders.unionByName(invalid_customer_orders)
)

print(
    "Orders after adding invalid references:",
    valid_orders_with_bad_references.count()
)

In [0]:
# Identify orders whose customer_id does not exist
# in the Silver Customers table

orders_with_invalid_customer = (
    valid_orders_with_bad_references
    .join(
        valid_customer_ids,
        on="customer_id",
        how="left_anti"
    )
)

print(
    "Orders with invalid customer references:",
    orders_with_invalid_customer.count()
)

display(orders_with_invalid_customer)

In [0]:
# Orders whose customer_id does not exist in Silver Customers
# are treated as invalid and will be sent to quarantine.

orders_with_invalid_customer = (
    valid_orders_with_bad_references
    .join(
        valid_customer_ids,
        on="customer_id",
        how="left_anti"
    )
)

# Keep only orders with valid customer references for Silver processing
orders_with_valid_customer = (
    valid_orders_with_bad_references
    .join(
        valid_customer_ids,
        on="customer_id",
        how="inner"
    )
)

print(
    "Orders with invalid customer references:",
    orders_with_invalid_customer.count()
)

print(
    "Orders with valid customer references:",
    orders_with_valid_customer.count()
)

In [0]:
# Define a window that groups records by order_id
# and places the most recently updated record first.
order_window = (
    Window
    .partitionBy("order_id")
    .orderBy(F.col("updated_at").desc())
)

# Assign a row number to each version of an order.
# The latest record receives row_num = 1.
orders_ranked = (
    orders_with_valid_customer
    .withColumn(
        "row_num",
        F.row_number().over(order_window)
    )
)

# Keep only the latest version of each order.
orders_deduped = (
    orders_ranked
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print("Orders after deduplication:", orders_deduped.count())

In [0]:
# Write clean, validated and deduplicated orders to the Silver layer.
# Silver contains only records that passed our data-quality checks.

orders_deduped.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver.orders")


# Combine all invalid orders into one quarantine dataset.
# This includes:
# 1. Orders with invalid order statuses
# 2. Orders with invalid customer references

orders_quarantine = (
    invalid_orders
    .unionByName(
        orders_with_invalid_customer
    )
)

print(
    "Total orders sent to quarantine:",
    orders_quarantine.count()
)


# Store rejected records separately so they can be investigated
# without contaminating the Silver layer.

orders_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_quarantine.orders")

In [0]:
# Read the persisted Silver Orders table
# and verify that the expected records were written.

silver_orders = spark.table(
    "retail_silver.orders"
)

print(
    "Silver Orders count:",
    silver_orders.count()
)

display(
    silver_orders.limit(10)
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load raw Order Items from the Bronze layer.
order_items_bronze = spark.table(
    "retail_bronze.order_items"
)

print(
    "Bronze Order Items:",
    order_items_bronze.count()
)

display(
    order_items_bronze.limit(10)
)

In [0]:
# Identify Order Items with invalid quantities.
# Quantity must be greater than zero for a valid order item.

invalid_quantity_items = (
    order_items_bronze
    .filter(
        F.col("quantity") <= 0
    )
)

print(
    "Order Items with invalid quantity:",
    invalid_quantity_items.count()
)

display(
    invalid_quantity_items
)

In [0]:
# Keep only records with valid quantities.
valid_quantity_items = (
    order_items_bronze
    .filter(
        F.col("quantity") > 0
    )
)

# Load valid Product IDs from the Silver Products table.
valid_product_ids = (
    spark.table("retail_silver.products")
    .select("product_id")
    .distinct()
)

# Find Order Items whose product_id does not exist
# in the Silver Products table.
items_with_invalid_product = (
    valid_quantity_items
    .join(
        valid_product_ids,
        on="product_id",
        how="left_anti"
    )
)

print(
    "Order Items with invalid product references:",
    items_with_invalid_product.count()
)

display(
    items_with_invalid_product.limit(20)
)

In [0]:
# Load raw Products from the Bronze layer.
products_bronze = spark.table(
    "retail_bronze.products"
)

print(
    "Bronze Products:",
    products_bronze.count()
)

display(
    products_bronze.limit(10)
)

# Identify products with missing product IDs.
invalid_product_id = (
    products_bronze
    .filter(F.col("product_id").isNull())
)

# Identify products with invalid prices.
invalid_product_price = (
    products_bronze
    .filter(F.col("price") <= 0)
)

print(
    "Products with null product_id:",
    invalid_product_id.count()
)

print(
    "Products with invalid price:",
    invalid_product_price.count()
)

In [0]:
# Write validated Products into the Silver layer.
# Since all records passed our quality checks, they can be promoted as-is.

products_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver.products")

print("Silver Products:", spark.table(
    "retail_silver.products"
).count())

In [0]:
# Keep only Order Items with a valid quantity.
valid_quantity_items = (
    order_items_bronze
    .filter(
        F.col("quantity") > 0
    )
)

# Load the valid Product IDs from the Silver Products table.
valid_product_ids = (
    spark.table("retail_silver.products")
    .select("product_id")
    .distinct()
)

# Find Order Items whose product_id does not exist
# in the Silver Products table.
items_with_invalid_product = (
    valid_quantity_items
    .join(
        valid_product_ids,
        on="product_id",
        how="left_anti"
    )
)

print(
    "Order Items with invalid product references:",
    items_with_invalid_product.count()
)

display(
    items_with_invalid_product.limit(20)
)

In [0]:
# Check all Order Items for invalid product references,
# without filtering on quantity first.

items_with_invalid_product = (
    order_items_bronze
    .join(
        valid_product_ids,
        on="product_id",
        how="left_anti"
    )
)

print(
    "Order Items with invalid product references:",
    items_with_invalid_product.count()
)

display(
    items_with_invalid_product.limit(20)
)

In [0]:
# Keep only Order Items that satisfy both quality rules:
# 1. Quantity must be greater than zero.
# 2. Product ID must exist in Silver Products.

valid_order_items = (
    order_items_bronze
    .filter(
        F.col("quantity") > 0
    )
    .join(
        valid_product_ids,
        on="product_id",
        how="inner"
    )
)

print(
    "Valid Order Items:",
    valid_order_items.count()
)

In [0]:
# Load valid Order IDs from the Silver Orders table.
valid_order_ids = (
    spark.table("retail_silver.orders")
    .select("order_id")
    .distinct()
)

# Find Order Items whose order_id does not exist
# in the Silver Orders table.
items_with_invalid_order = (
    valid_order_items
    .join(
        valid_order_ids,
        on="order_id",
        how="left_anti"
    )
)

print(
    "Order Items with invalid order references:",
    items_with_invalid_order.count()
)

display(
    items_with_invalid_order.limit(20)
)

In [0]:
# Define a window that groups records by order_item_id
# and puts the most recently updated version first.
order_item_window = (
    Window
    .partitionBy("order_item_id")
    .orderBy(F.col("updated_at").desc())
)

# Assign a row number to each version of an Order Item.
# The latest version receives row_num = 1.
order_items_ranked = (
    valid_order_items
    .withColumn(
        "row_num",
        F.row_number().over(order_item_window)
    )
)

# Keep only the latest version of each Order Item.
# row_num is a temporary helper column, so remove it afterward.
order_items_deduped = (
    order_items_ranked
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print(
    "Order Items after deduplication:",
    order_items_deduped.count()
)

In [0]:
# Write clean, validated and deduplicated Order Items
# into the Silver layer.

order_items_deduped.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_silver.order_items")


# Store the invalid Order Items separately for investigation.
# These are the records that failed our data-quality checks.

order_items_quarantine = (
    order_items_bronze
    .filter(
        (F.col("quantity") <= 0) |
        (~F.col("product_id").isin(
            valid_product_ids.select("product_id")
        ))
    )
)

print(
    "Order Items sent to quarantine:",
    order_items_quarantine.count()
)


# Persist rejected records in the quarantine layer.

order_items_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_quarantine.order_items")

In [0]:
# Read the persisted Silver Order Items table
# and verify that the expected records were written.

silver_order_items = spark.table(
    "retail_silver.order_items"
)

print(
    "Silver Order Items:",
    silver_order_items.count()
)

display(
    silver_order_items.limit(10)
)